In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest
from sklearn import model_selection
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

In [47]:
df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv") 

# 2. 最初の5行を見る
print(df.head())

# 3. データの傾向（平均とか最大とか）をまとめて出す
print(df.describe())

# 4. 行数・列数（（行, 列）の形で出ます）
print(df.shape)

  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4         True  
  

必要なpandasとファイルを読み込み、ファイルの最初の5行を表示しました　
傾向なども表示しました
どの程度の人が生き残ったのかまとめます：

In [48]:
train = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")
survival_rate = train['Transported'].mean()
print(f"全体の生存率（転送率）: {survival_rate:.2%}")

# 人数で確認する場合
print(train['Transported'].value_counts())

全体の生存率（転送率）: 50.36%
Transported
True     4378
False    4315
Name: count, dtype: int64


約半数が転送されたことが分かります。Cyrosleep（寝ていた人）は行動できません。これが転送とどう関係があるのかを調べるために、寝ていたという条件のもとで起きていた人の条件付き確率を表示します

In [49]:
cryo_sleep_people = train[train['CryoSleep'] == True]

cryo_survival_rate = cryo_sleep_people['Transported'].mean()

print(f"寝ていた人の生存率: {cryo_survival_rate:.2%}")

寝ていた人の生存率: 81.76%


寝ていた人の方が逃げづらい→転送されやすい　と思っていましたが、実際は逆で、寝ていた人は異常事態（次元の裂け目）が発生した際、個室の中にいたことで、むき出しのデッキにいた人よりも物理的な衝撃や影響から保護された可能性があります。
ですがこの議論は課題とは関係ないので、ここまでにしておきます
ここで欠損データの個数をかくにんします：

In [50]:
dataset = pd.concat([train, test], ignore_index = True)

PassengerId = test['PassengerId']

dataset_null = dataset.fillna(np.nan)
dataset_null.isnull().sum()

PassengerId        0
HomePlanet       288
CryoSleep        310
Cabin            299
Destination      274
Age              270
VIP              296
RoomService      263
FoodCourt        289
ShoppingMall     306
Spa              284
VRDeck           268
Name             294
Transported     4277
dtype: int64

passenger ID 意外は多くの欠損データがあります。ここで、欠損データを以下のルールで埋めます
・文字データは「unknown」として埋める
・寝ているか不明な人は一番多い起きていた人「False」として埋める
・年齢は平均値で埋める
・寝ている人のサービス利用料（'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'）はすべて0で埋める。起きている人は、平均値で埋める
ここで、自分出来アタコードはエラーが出てしまったのでGeminiに書いてもらいました


In [51]:
X = train.drop(['Transported', 'PassengerId', 'Name'], axis=1)
y = train['Transported']
# 1. Cabinを分解する（エラーが出ないように慎重に）
if 'Cabin' in train.columns:
    # Deck(階層), Num(部屋番号), Side(側)に分ける
    train[['Deck', 'Num', 'Side']] = train['Cabin'].str.split('/', expand=True)
    
    # Numは数字として扱うために変換（欠損値はとりあえず-1にする）
    train['Num'] = pd.to_numeric(train['Num'], errors='coerce').fillna(-1)
    
    # 文字データの欠損値を埋める
    train['Deck'] = train['Deck'].fillna('Unknown')
    train['Side'] = train['Side'].fillna('Unknown')
    
    # 不要になった元のCabinを消す
    train = train.drop('Cabin', axis=1)

# 2. 合計支出額（TotalSpent）を作る
services = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
train['TotalSpent'] = train[services].sum(axis=1)

# 3. 再度ワンホット・エンコーディング（新しく増えたDeckやSideも対象）
# すでにダミー化されている列を除外して実行
cols_to_encode = ['HomePlanet', 'Destination', 'Deck', 'Side']
# 存在する列だけをエンコード対象にする
cols_to_encode = [c for c in cols_to_encode if c in train.columns]
train = pd.get_dummies(train, columns=cols_to_encode)

# 4. 学習データの準備
# 文字列のまま残っている可能性がある列を除外
X = train.select_dtypes(exclude=['object']).drop(['Transported', 'PassengerId'], axis=1, errors='ignore')
y = train['Transported']

# 5. モデルの強化（パラメータを少し調整）
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(
    n_estimators=300,    # 木の数を増やして安定させる
    max_depth=8,        # 深さを制限して「暗記」を防ぐ
    min_samples_leaf=5,  # 1つの葉に最低5サンプル（ノイズ対策）
    random_state=42
)

model.fit(X_train, y_train)
print(f"現在の精度: {model.score(X_val, y_val):.2%}")


現在の精度: 79.01%


In [52]:
train['CryoSleep'] = train['CryoSleep'].fillna(train['CryoSleep'].mode()[0])

train['Age'] = train['Age'].fillna(train['Age'].mean())

services = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

for col in services:
    train.loc[train['CryoSleep'] == True, col] = train.loc[train['CryoSleep'] == True, col].fillna(0)

for col in services:
    awake_mean = train.loc[train['CryoSleep'] == False, col].mean()
    train.loc[train['CryoSleep'] == False, col] = train.loc[train['CryoSleep'] == False, col].fillna(awake_mean)

char_cols = ['HomePlanet', 'Destination', 'VIP']
for col in char_cols:
    train[col] = train[col].fillna(train[col].mode()[0])

print(train[['Age'] + services].isnull().sum())

KeyError: 'HomePlanet'

また、文字データを0と1に変換します

あ
